In [ ]:
"""
Extrae la serie de precipitación quincenal (total acumulado, días con
lluvia, y máximo diario) para un punto, usando CHIRPS Daily.

A diferencia de temperatura (que se promedia), precipitación se ACUMULA:
lo relevante agronómicamente es cuánta lluvia total cayó, no la tasa
diaria promedio.

CÓMO LEER EL RESULTADO
-----------------------
precip_total_mm         Lluvia total acumulada en la quincena, en mm.
precip_dias_lluvia       Cuántos días de la quincena tuvieron lluvia
                         (>1mm, umbral estándar para distinguir lluvia
                         real de humedad/rocío medida por error).
                         Relevante para el criterio de "no más de una
                         semana sin agua": una quincena puede tener el
                         mismo total_mm con distribución muy distinta
                         (ej. toda la lluvia en 2 días vs repartida en 8).
precip_max_diario_mm     El día más lluvioso de la quincena. Señal de
                         eventos de lluvia intensa (riesgo de erosión/
                         encharcamiento) que el total o los días de
                         lluvia por sí solos no muestran.

LIMITACIÓN A TENER EN CUENTA: precip_dias_lluvia cuenta días con lluvia
DENTRO de la quincena, pero no detecta rachas secas que crucen el borde
entre dos quincenas (ej. últimos 4 días secos de una quincena + primeros
5 días secos de la siguiente = 9 días secos seguidos, invisible en esta
tabla). Si el análisis de riego necesita capturar eso, se puede agregar
un cálculo de "racha seca máxima" sobre la serie diaria completa aparte.
"""
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee

from period_utils import build_biweekly_periods


def get_precipitation_biweekly(lat, lon, start_date="2016-01-01", end_date=None,
                                rain_threshold_mm=1.0):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
    rain_threshold_mm: umbral para considerar un día como "día de lluvia"
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    chirps_coll = (
        ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .select('precipitation')
        .filterDate(str(start), str(end + timedelta(days=1)))
    )

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = chirps_coll.filterDate(p_start, p_end)

        total = filtered.sum().rename('precip_total_mm')
        max_daily = filtered.max().rename('precip_max_diario_mm')
        rain_days = filtered.map(lambda img: img.gt(rain_threshold_mm)).sum().rename('precip_dias_lluvia')

        combined = total.addBands(max_daily).addBands(rain_days)

        stats = combined.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=5566,  # resolución nativa de CHIRPS
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            'lat': lat,
            'lon': lon,
            'precip_total_mm': round(props.get('precip_total_mm'), 2) if props.get('precip_total_mm') is not None else None,
            'precip_dias_lluvia': props.get('precip_dias_lluvia'),
            'precip_max_diario_mm': round(props.get('precip_max_diario_mm'), 2) if props.get('precip_max_diario_mm') is not None else None,
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    # CHIRPS suele tener 1-2 meses de rezago en publicar los datos más
    # recientes (necesita datos de estaciones terrestres para el producto
    # final) -> aunque una quincena ya "pasó" en el calendario, puede que
    # el proveedor todavía no la haya publicado. Avisamos si encontramos
    # filas nulas al final de la serie, en vez de dejarlas pasar en silencio.
    filas_nulas = df['precip_total_mm'].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df['precip_total_mm'].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin datos todavía (rezago de "
              f"publicación de CHIRPS): {primeras_nulas}")

    return df


def save_precipitation_profile(df, out_prefix="precipitation_biweekly", output_dir="../databases"):
    """
    Guarda la serie de precipitación con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que temperature_profile.py)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    LAT = -19.689669877950884
    LON = 147.22717515914223

    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    df = get_precipitation_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_precipitation_profile(df)
    print(df.head(10))